# PDD Mobility Scanner — Depth Estimation from Hike Photos

Uses [Depth Anything V2](https://huggingface.co/depth-anything/Depth-Anything-V2-Small-hf) to extract monocular depth maps from trail photos captured by the scanner.

**Requirements:** GPU runtime recommended (Runtime → Change runtime type → T4 GPU).

**Upload your hike `.jpg` images when prompted.**

## Step 1: Install dependencies

In [ ]:
!pip install -q transformers torch pillow matplotlib

## Step 2: Upload hike photos

In [ ]:
from google.colab import files

uploaded = files.upload()
image_files = sorted(uploaded.keys())
print(f"Uploaded {len(image_files)} image(s): {image_files}")

## Step 3: Load the depth estimation model

We use the `depth-estimation` pipeline with **Depth Anything V2 Small** — a good balance of speed and quality.

In [ ]:
from transformers import pipeline
from PIL import Image

pipe = pipeline(task="depth-estimation", model="depth-anything/Depth-Anything-V2-Small-hf")
print("Model loaded.")

## Step 4: Run depth estimation on each image

In [ ]:
import numpy as np

results = []
for fname in image_files:
    image = Image.open(fname).convert("RGB")
    depth_output = pipe(image)
    depth_map = depth_output["depth"]  # PIL Image
    depth_array = np.array(depth_map)
    results.append({
        "filename": fname,
        "image": image,
        "depth_map": depth_map,
        "depth_array": depth_array,
    })
    print(f"{fname}: depth range [{depth_array.min()}, {depth_array.max()}], shape {depth_array.shape}")

print(f"\nProcessed {len(results)} image(s).")

## Step 5: Visualize results

Side-by-side view of each original photo and its depth map.

In [ ]:
import matplotlib.pyplot as plt

for r in results:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].imshow(r["image"])
    axes[0].set_title(r["filename"])
    axes[0].axis("off")

    im = axes[1].imshow(r["depth_array"], cmap="inferno")
    axes[1].set_title("Depth map")
    axes[1].axis("off")
    fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()

## Step 6: Depth statistics per image

In [ ]:
import pandas as pd

stats = []
for r in results:
    d = r["depth_array"].astype(float)
    stats.append({
        "filename": r["filename"],
        "depth_min": d.min(),
        "depth_max": d.max(),
        "depth_mean": d.mean(),
        "depth_std": d.std(),
        "depth_median": np.median(d),
    })

df_stats = pd.DataFrame(stats)
df_stats

## Step 7: Export depth maps and stats

In [ ]:
# Save each depth map as a PNG
for r in results:
    out_name = r["filename"].rsplit(".", 1)[0] + "_depth.png"
    r["depth_map"].save(out_name)
    print(f"Saved {out_name}")

# Save summary stats
df_stats.to_csv("depth_stats.csv", index=False)
print("Saved depth_stats.csv")

# Download
files.download("depth_stats.csv")
for r in results:
    out_name = r["filename"].rsplit(".", 1)[0] + "_depth.png"
    files.download(out_name)